# CSV → Final Cut Pro XML

아래 셀을 **1번부터 5번까지 순서대로 한 번씩** 실행하세요.

입력 파일은 다음 두 종류이며 서로 다른 위치에 저장됩니다.

```text
my-video/
├── timeline.csv   ← 편집 순서를 적은 CSV
└── Media/         ← CSV에 적은 사진과 영상
```

> CSV는 Media 폴더에 넣지 않습니다. 2번에서 CSV를, 3번에서 사진·영상을 따로 선택합니다.

## 1. 실행 코드 준비

GitHub에서 변환 코드를 내려받고 FFmpeg를 확인합니다. 이 셀에서는 사용자 파일을 업로드하지 않습니다.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

repo = Path('/content/csv-to-fcpxml-starter')
if (repo / '.git').is_dir():
    # 셀을 다시 실행한 경우 기존 폴더를 지우지 않고 최신 코드만 받습니다.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/Kongdataif/csv-to-fcpxml-starter.git',
        str(repo),
    ], check=True)

os.chdir(repo)
if shutil.which('ffprobe') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
print(f'준비 완료: {repo}')

## 2. CSV 한 개 업로드

- 이 버튼에서는 **CSV 파일 한 개만** 선택하세요. 사진과 영상은 다음 단계에서 선택합니다.
- 파일명은 달라도 괜찮지만 확장자는 반드시 `.csv`여야 합니다.
- 선택한 파일은 자동으로 `my-video/timeline.csv`에 저장됩니다.
- CSV의 `파일` 열에는 다음 단계에서 올릴 실제 미디어 파일명을 적어야 합니다.

In [ ]:
from google.colab import files
from pathlib import Path

project = Path('my-video')
project.mkdir(parents=True, exist_ok=True)

# 업로드 창에서 CSV 한 개만 선택합니다.
csv_upload = files.upload()
if len(csv_upload) != 1:
    raise ValueError('CSV 파일을 정확히 한 개만 선택해주세요.')

csv_name, csv_data = next(iter(csv_upload.items()))
if Path(csv_name).suffix.lower() != '.csv':
    raise ValueError(f'CSV 파일이 아닙니다: {csv_name}')

csv_target = project / 'timeline.csv'
csv_target.write_bytes(csv_data)
print(f'CSV 저장 완료: {csv_name} → {csv_target}')

## 3. 사진·영상 업로드

- 이 버튼에서는 CSV에 적은 **사진과 영상을 모두 선택**하세요. 여러 파일을 한 번에 선택할 수 있습니다.
- 선택한 파일은 `my-video/Media/`에 저장됩니다.
- CSV의 `파일` 값과 실제 파일명은 확장자까지 같아야 합니다. 예: `intro.mov`
- 지원 영상: MOV, MP4, M4V, MKV, AVI / 지원 사진: JPG, JPEG, PNG, HEIC, TIF, TIFF

In [ ]:
from google.colab import files
from pathlib import Path

media_dir = Path('my-video/Media')
media_dir.mkdir(parents=True, exist_ok=True)
supported = {'.mov', '.mp4', '.m4v', '.mkv', '.avi', '.jpg', '.jpeg', '.png', '.heic', '.tif', '.tiff'}

# 업로드 창에서 CSV에 적은 사진과 영상을 모두 선택합니다.
media_upload = files.upload()
invalid = [name for name in media_upload if Path(name).suffix.lower() not in supported]
if invalid:
    raise ValueError('지원하지 않는 파일입니다: ' + ', '.join(invalid))
if not media_upload:
    raise ValueError('사진이나 영상을 한 개 이상 선택해주세요.')

guide_file = media_dir / '여기에_사진과_영상을_넣으세요.txt'
guide_file.unlink(missing_ok=True)
for name, data in media_upload.items():
    (media_dir / Path(name).name).write_bytes(data)

print(f'미디어 {len(media_upload)}개 저장 완료: {media_dir}')
for path in sorted(media_dir.iterdir()):
    print(' -', path.name)

## 4. 출력 설정 후 변환

| 파라미터 | 값 | 의미 |
|---|---|---|
| `LAYOUT` | `portrait` | 세로 9:16, 1080×1920 |
| `LAYOUT` | `landscape` | 가로 16:9, 1920×1080 |
| `LAYOUT` | `both` | 세로와 가로를 모두 생성 |
| `FIT` | `fit` | 원본 전체 표시. 비율이 다르면 여백 가능 |
| `FIT` | `fill` | 화면을 채움. 원본 가장자리 잘림 가능 |

아래 드롭다운에서 값을 고른 뒤 셀을 실행하세요. 결과는 `my-video/output/`에 생성됩니다.

In [ ]:
from pathlib import Path
import subprocess

LAYOUT = 'portrait'  # @param ['portrait', 'landscape', 'both']
FIT = 'fit'  # @param ['fit', 'fill']

# 선택한 값만 명령어 인자로 전달합니다.
subprocess.run([
    'python3', 'run.py', 'my-video',
    '--layout', LAYOUT,
    '--fit', FIT,
], check=True)

print('생성 결과:')
for path in sorted(Path('my-video/output').glob('*.fcpxml')):
    print(' -', path)

## 5. 결과 ZIP 다운로드

FCPXML과 그 파일이 참조하는 미디어를 함께 ZIP으로 묶어 다운로드합니다. 원본 영상이 크면 ZIP도 커질 수 있습니다.

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive('/content/fcpxml-result', 'zip', 'my-video')
print(f'다운로드 준비 완료: {archive}')
files.download(archive)